<div style="display: flex; justify-content: flex-start; align-items: center;">
    <a href="https://colab.research.google.com/github/msfasha/307304-Data-Mining/blob/main/20251/Module%206-Time%20Series%20Analysis/time_series_analysis.ipynb" target="_blank">    
        <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab" style="height: 25px; margin-right: 20px;">
    </a>
</div>

# Part 3: Machine Learning Methods for Time Series

<div style="
    background-color:#8F0177;
    padding:15px;
    border-radius:8px;
    color:white;
    display:flex;
    align-items:center;
">
    <h3 style="margin:0;">Time Series Analysis in Python</h3>
</div>


<div style="
    background-color:#8F0177;
    padding:15px;
    border-radius:8px;
    color:white;
    display:flex;
    align-items:center;
">
    <h3 style="margin:0;">Setup and Imports</h3>
</div>

Before we begin, install the required packages:

```bash
pip install pandas numpy matplotlib seaborn scikit-learn xgboost lightgbm tensorflow prophet
```

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta

# Scikit-learn
from sklearn.model_selection import train_test_split, TimeSeriesSplit
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Gradient Boosting libraries
import xgboost as xgb
import lightgbm as lgb

import warnings
warnings.filterwarnings('ignore')

# Set plotting style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')

# Random seed for reproducibility
np.random.seed(42)

---

# 1. Feature Engineering for Time Series

<div style="
    background-color:#8F0177;
    padding:15px;
    border-radius:8px;
    color:white;
    display:flex;
    align-items:center;
">
    <h3 style="margin:0;">Why Feature Engineering?</h3>
</div>

Most ML algorithms don't understand time inherently. We need to:
- Convert temporal patterns into numeric features
- Create lag features to capture autocorrelation
- Extract date/time components
- Engineer domain-specific features

<div style="
    background-color:#8F0177;
    padding:15px;
    border-radius:8px;
    color:white;
    display:flex;
    align-items:center;
">
    <h3 style="margin:0;">1.1 Creating Sample Dataset</h3>
</div>


In [ ]:
# Generate synthetic time series data
np.random.seed(42)
dates = pd.date_range(start='2021-01-01', end='2023-12-31', freq='D')
n = len(dates)

# Components
trend = np.linspace(100, 200, n)
yearly_seasonality = 20 * np.sin(2 * np.pi * np.arange(n) / 365.25)
weekly_seasonality = 10 * np.sin(2 * np.pi * np.arange(n) / 7)
noise = np.random.normal(0, 5, n)

# Target variable
sales = trend + yearly_seasonality + weekly_seasonality + noise

# Create DataFrame
df = pd.DataFrame({
    'sales': sales,
    'temperature': 15 + 10 * np.sin(2 * np.pi * np.arange(n) / 365.25) + np.random.normal(0, 3, n),
    'promotion': np.random.choice([0, 1], size=n, p=[0.8, 0.2])
}, index=dates)

print("Dataset shape:", df.shape)
print("\nFirst 10 rows:")
print(df.head(10))
print("\nDataset info:")
print(df.info())

In [ ]:
# Visualize the data
fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=True)

df['sales'].plot(ax=axes[0], linewidth=1, color='steelblue')
axes[0].set_title('Sales Over Time', fontsize=12, fontweight='bold')
axes[0].set_ylabel('Sales', fontsize=11)
axes[0].grid(True, alpha=0.3)

df['temperature'].plot(ax=axes[1], linewidth=1, color='orange')
axes[1].set_title('Temperature', fontsize=12, fontweight='bold')
axes[1].set_ylabel('Temperature', fontsize=11)
axes[1].grid(True, alpha=0.3)

df['promotion'].plot(ax=axes[2], linewidth=0.5, color='green', alpha=0.7)
axes[2].set_title('Promotion (Binary)', fontsize=12, fontweight='bold')
axes[2].set_ylabel('Promotion', fontsize=11)
axes[2].set_xlabel('Date', fontsize=11)
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

<div style="
    background-color:#8F0177;
    padding:15px;
    border-radius:8px;
    color:white;
    display:flex;
    align-items:center;
">
    <h3 style="margin:0;">1.2 Lag Features</h3>
</div>

Create features from past values to capture autocorrelation:

In [ ]:
def create_lag_features(df, target_col, lags):
    """
    Create lag features for specified lags
    """
    df_copy = df.copy()
    for lag in lags:
        df_copy[f'{target_col}_lag{lag}'] = df_copy[target_col].shift(lag)
    return df_copy

# Create lag features: 1, 2, 3 days and 1, 2 weeks
lags = [1, 2, 3, 7, 14]
df_features = create_lag_features(df, 'sales', lags)

print("DataFrame with lag features:")
print(df_features.head(20))
print(f"\nShape: {df_features.shape}")

<div style="
    background-color:#8F0177;
    padding:15px;
    border-radius:8px;
    color:white;
    display:flex;
    align-items:center;
">
    <h3 style="margin:0;">1.3 Rolling Window Features</h3>
</div>

Statistical features computed over rolling windows:

In [ ]:
def create_rolling_features(df, target_col, windows):
    """
    Create rolling statistical features
    """
    df_copy = df.copy()
    for window in windows:
        # Rolling mean
        df_copy[f'{target_col}_rolling_mean_{window}'] = \
            df_copy[target_col].rolling(window=window).mean()
        
        # Rolling std (volatility)
        df_copy[f'{target_col}_rolling_std_{window}'] = \
            df_copy[target_col].rolling(window=window).std()
        
        # Rolling min/max
        df_copy[f'{target_col}_rolling_min_{window}'] = \
            df_copy[target_col].rolling(window=window).min()
        df_copy[f'{target_col}_rolling_max_{window}'] = \
            df_copy[target_col].rolling(window=window).max()
    
    return df_copy

# Add rolling features for 7 and 14 day windows
windows = [7, 14]
df_features = create_rolling_features(df_features, 'sales', windows)

print("DataFrame with rolling features:")
print(df_features.iloc[20:25])
print(f"\nTotal columns: {df_features.shape[1]}")

<div style="
    background-color:#8F0177;
    padding:15px;
    border-radius:8px;
    color:white;
    display:flex;
    align-items:center;
">
    <h3 style="margin:0;">1.4 Date/Time Features</h3>
</div>

Extract temporal information from the datetime index:

In [ ]:
def create_date_features(df):
    """
    Extract date/time features from DatetimeIndex
    """
    df_copy = df.copy()
    
    # Basic date features
    df_copy['day_of_week'] = df_copy.index.dayofweek
    df_copy['day_of_month'] = df_copy.index.day
    df_copy['week_of_year'] = df_copy.index.isocalendar().week.astype(int)
    df_copy['month'] = df_copy.index.month
    df_copy['quarter'] = df_copy.index.quarter
    df_copy['year'] = df_copy.index.year
    
    # Binary features
    df_copy['is_weekend'] = (df_copy.index.dayofweek >= 5).astype(int)
    df_copy['is_month_start'] = df_copy.index.is_month_start.astype(int)
    df_copy['is_month_end'] = df_copy.index.is_month_end.astype(int)
    df_copy['is_quarter_start'] = df_copy.index.is_quarter_start.astype(int)
    df_copy['is_quarter_end'] = df_copy.index.is_quarter_end.astype(int)
    
    return df_copy

df_features = create_date_features(df_features)

print("Date/time features:")
date_cols = ['day_of_week', 'day_of_month', 'week_of_year', 'month', 
             'quarter', 'year', 'is_weekend']
print(df_features[date_cols].head(10))

<div style="
    background-color:#8F0177;
    padding:15px;
    border-radius:8px;
    color:white;
    display:flex;
    align-items:center;
">
    <h3 style="margin:0;">1.5 Cyclical Encoding</h3>
</div>

Encode cyclical features (day of week, month) using sine/cosine transformation:

In [ ]:
def encode_cyclical_features(df, col, max_val):
    """
    Encode cyclical feature using sine and cosine
    """
    df_copy = df.copy()
    df_copy[f'{col}_sin'] = np.sin(2 * np.pi * df_copy[col] / max_val)
    df_copy[f'{col}_cos'] = np.cos(2 * np.pi * df_copy[col] / max_val)
    return df_copy

# Encode cyclical features
df_features = encode_cyclical_features(df_features, 'day_of_week', 7)
df_features = encode_cyclical_features(df_features, 'month', 12)
df_features = encode_cyclical_features(df_features, 'day_of_month', 31)

print("Cyclical encoding:")
print(df_features[['day_of_week', 'day_of_week_sin', 'day_of_week_cos', 
                   'month', 'month_sin', 'month_cos']].head(10))

<div style="
    background-color:#8F0177;
    padding:15px;
    border-radius:8px;
    color:white;
    display:flex;
    align-items:center;
">
    <h3 style="margin:0;">1.6 Difference and Change Features</h3>
</div>


In [ ]:
# First difference
df_features['sales_diff1'] = df_features['sales'].diff(1)

# Percentage change
df_features['sales_pct_change'] = df_features['sales'].pct_change(1)

# Week-over-week change
df_features['sales_diff7'] = df_features['sales'].diff(7)

print("Change features:")
print(df_features[['sales', 'sales_diff1', 'sales_pct_change', 'sales_diff7']].head(15))

<div style="
    background-color:#8F0177;
    padding:15px;
    border-radius:8px;
    color:white;
    display:flex;
    align-items:center;
">
    <h3 style="margin:0;">1.7 Final Feature Summary</h3>
</div>


In [ ]:
print("\nTotal features created:")
print(f"Shape: {df_features.shape}")
print(f"\nColumn names:")
print(df_features.columns.tolist())

# Check for missing values (due to lags and rolling windows)
print("\nMissing values per column:")
missing = df_features.isnull().sum()
print(missing[missing > 0].sort_values(ascending=False))

---

# 2. Preparing Data for Machine Learning

<div style="
    background-color:#8F0177;
    padding:15px;
    border-radius:8px;
    color:white;
    display:flex;
    align-items:center;
">
    <h3 style="margin:0;">2.1 Train-Test Split for Time Series</h3>
</div>

**Important**: Use chronological splits, not random splits!

In [ ]:
# Drop rows with NaN (from lags and rolling features)
df_ml = df_features.dropna().copy()

print(f"Dataset after removing NaN: {df_ml.shape}")

# Define target and features
target = 'sales'
features = [col for col in df_ml.columns if col != target]

X = df_ml[features]
y = df_ml[target]

# Chronological split: 80% train, 20% test
split_idx = int(len(df_ml) * 0.8)

X_train = X.iloc[:split_idx]
X_test = X.iloc[split_idx:]
y_train = y.iloc[:split_idx]
y_test = y.iloc[split_idx:]

print(f"\nTrain set: {X_train.shape}")
print(f"Test set: {X_test.shape}")
print(f"\nTrain period: {X_train.index[0]} to {X_train.index[-1]}")
print(f"Test period: {X_test.index[0]} to {X_test.index[-1]}")

<div style="
    background-color:#8F0177;
    padding:15px;
    border-radius:8px;
    color:white;
    display:flex;
    align-items:center;
">
    <h3 style="margin:0;">2.2 Feature Scaling</h3>
</div>

Normalize features for models that are sensitive to scale:

In [ ]:
# Fit scaler on training data only
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Convert back to DataFrame for easier handling
X_train_scaled = pd.DataFrame(X_train_scaled, 
                              columns=X_train.columns, 
                              index=X_train.index)
X_test_scaled = pd.DataFrame(X_test_scaled, 
                             columns=X_test.columns, 
                             index=X_test.index)

print("Scaled features (first 5 rows):")
print(X_train_scaled.head())

---

# 3. Traditional Machine Learning Models

<div style="
    background-color:#8F0177;
    padding:15px;
    border-radius:8px;
    color:white;
    display:flex;
    align-items:center;
">
    <h3 style="margin:0;">3.1 Linear Regression</h3>
</div>

Simple baseline model:

In [ ]:
# Train Linear Regression
lr_model = LinearRegression()
lr_model.fit(X_train_scaled, y_train)

# Predictions
y_pred_lr_train = lr_model.predict(X_train_scaled)
y_pred_lr_test = lr_model.predict(X_test_scaled)

# Evaluate
def evaluate_model(y_true, y_pred, model_name, dataset='Test'):
    """
    Calculate and print evaluation metrics
    """
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2 = r2_score(y_true, y_pred)
    mape = np.mean(np.abs((y_true - y_pred) / y_true)) * 100
    
    print(f"\n{model_name} - {dataset} Set:")
    print(f"  MAE:  {mae:.2f}")
    print(f"  RMSE: {rmse:.2f}")
    print(f"  R²:   {r2:.4f}")
    print(f"  MAPE: {mape:.2f}%")
    
    return {'Model': model_name, 'Dataset': dataset, 'MAE': mae, 'RMSE': rmse, 'R2': r2, 'MAPE': mape}

results = []
results.append(evaluate_model(y_train, y_pred_lr_train, 'Linear Regression', 'Train'))
results.append(evaluate_model(y_test, y_pred_lr_test, 'Linear Regression', 'Test'))

<div style="
    background-color:#8F0177;
    padding:15px;
    border-radius:8px;
    color:white;
    display:flex;
    align-items:center;
">
    <h3 style="margin:0;">3.2 Regularized Linear Models (Ridge & Lasso)</h3>
</div>


In [ ]:
# Ridge Regression (L2 regularization)
ridge_model = Ridge(alpha=1.0)
ridge_model.fit(X_train_scaled, y_train)
y_pred_ridge = ridge_model.predict(X_test_scaled)
results.append(evaluate_model(y_test, y_pred_ridge, 'Ridge Regression', 'Test'))

# Lasso Regression (L1 regularization)
lasso_model = Lasso(alpha=0.1)
lasso_model.fit(X_train_scaled, y_train)
y_pred_lasso = lasso_model.predict(X_test_scaled)
results.append(evaluate_model(y_test, y_pred_lasso, 'Lasso Regression', 'Test'))

<div style="
    background-color:#8F0177;
    padding:15px;
    border-radius:8px;
    color:white;
    display:flex;
    align-items:center;
">
    <h3 style="margin:0;">3.3 Decision Tree</h3>
</div>


In [ ]:
# Decision Tree (no scaling needed)
dt_model = DecisionTreeRegressor(max_depth=10, min_samples_split=20, random_state=42)
dt_model.fit(X_train, y_train)
y_pred_dt = dt_model.predict(X_test)
results.append(evaluate_model(y_test, y_pred_dt, 'Decision Tree', 'Test'))

# Feature importance
feature_importance = pd.DataFrame({
    'feature': X_train.columns,
    'importance': dt_model.feature_importances_
}).sort_values('importance', ascending=False)

print("\nTop 10 Most Important Features:")
print(feature_importance.head(10))

<div style="
    background-color:#8F0177;
    padding:15px;
    border-radius:8px;
    color:white;
    display:flex;
    align-items:center;
">
    <h3 style="margin:0;">3.4 Random Forest</h3>
</div>


In [ ]:
# Random Forest
rf_model = RandomForestRegressor(
    n_estimators=100,
    max_depth=15,
    min_samples_split=10,
    min_samples_leaf=5,
    random_state=42,
    n_jobs=-1
)

rf_model.fit(X_train, y_train)
y_pred_rf = rf_model.predict(X_test)
results.append(evaluate_model(y_test, y_pred_rf, 'Random Forest', 'Test'))

# Feature importance
rf_importance = pd.DataFrame({
    'feature': X_train.columns,
    'importance': rf_model.feature_importances_
}).sort_values('importance', ascending=False)

print("\nRandom Forest - Top 10 Features:")
print(rf_importance.head(10))

In [ ]:
# Visualize feature importance
fig, ax = plt.subplots(figsize=(10, 6))
rf_importance.head(15).plot(x='feature', y='importance', kind='barh', ax=ax, color='steelblue')
ax.set_title('Random Forest - Top 15 Feature Importances', fontsize=12, fontweight='bold')
ax.set_xlabel('Importance', fontsize=11)
ax.set_ylabel('Feature', fontsize=11)
plt.tight_layout()
plt.show()

<div style="
    background-color:#8F0177;
    padding:15px;
    border-radius:8px;
    color:white;
    display:flex;
    align-items:center;
">
    <h3 style="margin:0;">3.5 Gradient Boosting (Scikit-learn)</h3>
</div>


In [ ]:
# Gradient Boosting Regressor
gb_model = GradientBoostingRegressor(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=5,
    min_samples_split=10,
    random_state=42
)

gb_model.fit(X_train, y_train)
y_pred_gb = gb_model.predict(X_test)
results.append(evaluate_model(y_test, y_pred_gb, 'Gradient Boosting', 'Test'))

<div style="
    background-color:#8F0177;
    padding:15px;
    border-radius:8px;
    color:white;
    display:flex;
    align-items:center;
">
    <h3 style="margin:0;">3.6 XGBoost</h3>
</div>


In [ ]:
# XGBoost
xgb_model = xgb.XGBRegressor(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=6,
    min_child_weight=3,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)

xgb_model.fit(X_train, y_train)
y_pred_xgb = xgb_model.predict(X_test)
results.append(evaluate_model(y_test, y_pred_xgb, 'XGBoost', 'Test'))

<div style="
    background-color:#8F0177;
    padding:15px;
    border-radius:8px;
    color:white;
    display:flex;
    align-items:center;
">
    <h3 style="margin:0;">3.7 LightGBM</h3>
</div>


In [ ]:
# LightGBM
lgb_model = lgb.LGBMRegressor(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=6,
    num_leaves=31,
    min_child_samples=20,
    random_state=42,
    verbose=-1
)

lgb_model.fit(X_train, y_train)
y_pred_lgb = lgb_model.predict(X_test)
results.append(evaluate_model(y_test, y_pred_lgb, 'LightGBM', 'Test'))

<div style="
    background-color:#8F0177;
    padding:15px;
    border-radius:8px;
    color:white;
    display:flex;
    align-items:center;
">
    <h3 style="margin:0;">3.8 Model Comparison</h3>
</div>


In [ ]:
# Create results DataFrame
results_df = pd.DataFrame(results)
test_results = results_df[results_df['Dataset'] == 'Test'].sort_values('RMSE')

print("\n" + "="*80)
print("MODEL COMPARISON - TEST SET")
print("="*80)
print(test_results.to_string(index=False))
print("\n" + "="*80)
print(f"Best Model: {test_results.iloc[0]['Model']} (RMSE: {test_results.iloc[0]['RMSE']:.2f})")
print("="*80)

In [ ]:
# Visualize model comparison
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# RMSE comparison
test_results.plot(x='Model', y='RMSE', kind='barh', ax=axes[0], color='steelblue', legend=False)
axes[0].set_title('Model Comparison - RMSE (Lower is Better)', fontsize=12, fontweight='bold')
axes[0].set_xlabel('RMSE', fontsize=11)
axes[0].set_ylabel('Model', fontsize=11)

# R² comparison
test_results.plot(x='Model', y='R2', kind='barh', ax=axes[1], color='orange', legend=False)
axes[1].set_title('Model Comparison - R² (Higher is Better)', fontsize=12, fontweight='bold')
axes[1].set_xlabel('R² Score', fontsize=11)
axes[1].set_ylabel('Model', fontsize=11)

plt.tight_layout()
plt.show()

<div style="
    background-color:#8F0177;
    padding:15px;
    border-radius:8px;
    color:white;
    display:flex;
    align-items:center;
">
    <h3 style="margin:0;">3.9 Visualizing Predictions</h3>
</div>


In [ ]:
# Compare best models visually
fig, ax = plt.subplots(figsize=(14, 6))

# Plot actual
y_test.plot(ax=ax, label='Actual', linewidth=2, color='black', alpha=0.7)

# Plot predictions from best models
pd.Series(y_pred_rf, index=y_test.index).plot(ax=ax, label='Random Forest', 
                                                linewidth=1.5, linestyle='--', alpha=0.8)
pd.Series(y_pred_xgb, index=y_test.index).plot(ax=ax, label='XGBoost', 
                                                 linewidth=1.5, linestyle='--', alpha=0.8)
pd.Series(y_pred_lgb, index=y_test.index).plot(ax=ax, label='LightGBM', 
                                                 linewidth=1.5, linestyle='--', alpha=0.8)

ax.set_title('Model Predictions vs Actual (Test Set)', fontsize=14, fontweight='bold')
ax.set_xlabel('Date', fontsize=12)
ax.set_ylabel('Sales', fontsize=12)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

---

# 4. Deep Learning for Time Series

<div style="
    background-color:#8F0177;
    padding:15px;
    border-radius:8px;
    color:white;
    display:flex;
    align-items:center;
">
    <h3 style="margin:0;">4.1 Data Preparation for Deep Learning</h3>
</div>

Create sequences for LSTM/GRU models:

In [ ]:
def create_sequences(data, seq_length):
    """
    Create sequences for LSTM/GRU
    """
    X, y = [], []
    for i in range(len(data) - seq_length):
        X.append(data[i:i+seq_length])
        y.append(data[i+seq_length])
    return np.array(X), np.array(y)

# Prepare data for LSTM
# Use only sales and temperature for simplicity
data_dl = df[['sales', 'temperature']].values

# Normalize
scaler_dl = StandardScaler()
data_scaled = scaler_dl.fit_transform(data_dl)

# Create sequences (30 days lookback)
seq_length = 30
X_seq, y_seq = create_sequences(data_scaled[:, 0], seq_length)  # Only sales for now

# Split
train_size = int(len(X_seq) * 0.8)
X_train_seq = X_seq[:train_size]
X_test_seq = X_seq[train_size:]
y_train_seq = y_seq[:train_size]
y_test_seq = y_seq[train_size:]

# Reshape for LSTM [samples, timesteps, features]
X_train_seq = X_train_seq.reshape((X_train_seq.shape[0], X_train_seq.shape[1], 1))
X_test_seq = X_test_seq.reshape((X_test_seq.shape[0], X_test_seq.shape[1], 1))

print(f"Sequence data shape:")
print(f"X_train: {X_train_seq.shape}")
print(f"y_train: {y_train_seq.shape}")
print(f"X_test: {X_test_seq.shape}")
print(f"y_test: {y_test_seq.shape}")

<div style="
    background-color:#8F0177;
    padding:15px;
    border-radius:8px;
    color:white;
    display:flex;
    align-items:center;
">
    <h3 style="margin:0;">4.2 LSTM Model</h3>
</div>


In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

# Set random seed
tf.random.set_seed(42)

# Build LSTM model
lstm_model = keras.Sequential([
    layers.LSTM(64, activation='relu', return_sequences=True, input_shape=(seq_length, 1)),
    layers.Dropout(0.2),
    layers.LSTM(32, activation='relu'),
    layers.Dropout(0.2),
    layers.Dense(16, activation='relu'),
    layers.Dense(1)
])

lstm_model.compile(optimizer='adam', loss='mse', metrics=['mae'])

print("LSTM Model Architecture:")
lstm_model.summary()

In [ ]:
# Train LSTM
early_stopping = keras.callbacks.EarlyStopping(
    monitor='val_loss',
    patience=10,
    restore_best_weights=True
)

history = lstm_model.fit(
    X_train_seq, y_train_seq,
    epochs=50,
    batch_size=32,
    validation_split=0.2,
    callbacks=[early_stopping],
    verbose=0
)

print("\nLSTM Training Complete!")
print(f"Best epoch: {early_stopping.best_epoch + 1}")
print(f"Best validation loss: {min(history.history['val_loss']):.4f}")

In [ ]:
# Plot training history
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Loss
axes[0].plot(history.history['loss'], label='Train Loss')
axes[0].plot(history.history['val_loss'], label='Validation Loss')
axes[0].set_title('LSTM Training History - Loss', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Epoch', fontsize=11)
axes[0].set_ylabel('Loss', fontsize=11)
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# MAE
axes[1].plot(history.history['mae'], label='Train MAE')
axes[1].plot(history.history['val_mae'], label='Validation MAE')
axes[1].set_title('LSTM Training History - MAE', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Epoch', fontsize=11)
axes[1].set_ylabel('MAE', fontsize=11)
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Evaluate LSTM
y_pred_lstm = lstm_model.predict(X_test_seq, verbose=0)

# Inverse transform predictions
# Create dummy array for inverse transform
dummy = np.zeros((len(y_pred_lstm), 2))
dummy[:, 0] = y_pred_lstm.flatten()
y_pred_lstm_original = scaler_dl.inverse_transform(dummy)[:, 0]

dummy[:, 0] = y_test_seq
y_test_lstm_original = scaler_dl.inverse_transform(dummy)[:, 0]

# Evaluate
lstm_mae = mean_absolute_error(y_test_lstm_original, y_pred_lstm_original)
lstm_rmse = np.sqrt(mean_squared_error(y_test_lstm_original, y_pred_lstm_original))
lstm_r2 = r2_score(y_test_lstm_original, y_pred_lstm_original)

print("\nLSTM Model - Test Set:")
print(f"  MAE:  {lstm_mae:.2f}")
print(f"  RMSE: {lstm_rmse:.2f}")
print(f"  R²:   {lstm_r2:.4f}")

<div style="
    background-color:#8F0177;
    padding:15px;
    border-radius:8px;
    color:white;
    display:flex;
    align-items:center;
">
    <h3 style="margin:0;">4.3 GRU Model</h3>
</div>

GRU is simpler than LSTM, often trains faster:

In [ ]:
# Build GRU model
gru_model = keras.Sequential([
    layers.GRU(64, activation='relu', return_sequences=True, input_shape=(seq_length, 1)),
    layers.Dropout(0.2),
    layers.GRU(32, activation='relu'),
    layers.Dropout(0.2),
    layers.Dense(16, activation='relu'),
    layers.Dense(1)
])

gru_model.compile(optimizer='adam', loss='mse', metrics=['mae'])

# Train GRU
history_gru = gru_model.fit(
    X_train_seq, y_train_seq,
    epochs=50,
    batch_size=32,
    validation_split=0.2,
    callbacks=[early_stopping],
    verbose=0
)

# Evaluate
y_pred_gru = gru_model.predict(X_test_seq, verbose=0)
dummy[:, 0] = y_pred_gru.flatten()
y_pred_gru_original = scaler_dl.inverse_transform(dummy)[:, 0]

gru_mae = mean_absolute_error(y_test_lstm_original, y_pred_gru_original)
gru_rmse = np.sqrt(mean_squared_error(y_test_lstm_original, y_pred_gru_original))
gru_r2 = r2_score(y_test_lstm_original, y_pred_gru_original)

print("\nGRU Model - Test Set:")
print(f"  MAE:  {gru_mae:.2f}")
print(f"  RMSE: {gru_rmse:.2f}")
print(f"  R²:   {gru_r2:.4f}")

<div style="
    background-color:#8F0177;
    padding:15px;
    border-radius:8px;
    color:white;
    display:flex;
    align-items:center;
">
    <h3 style="margin:0;">4.4 1D CNN Model</h3>
</div>


In [ ]:
# Build 1D CNN model
cnn_model = keras.Sequential([
    layers.Conv1D(filters=64, kernel_size=3, activation='relu', input_shape=(seq_length, 1)),
    layers.MaxPooling1D(pool_size=2),
    layers.Conv1D(filters=32, kernel_size=3, activation='relu'),
    layers.MaxPooling1D(pool_size=2),
    layers.Flatten(),
    layers.Dense(50, activation='relu'),
    layers.Dropout(0.2),
    layers.Dense(1)
])

cnn_model.compile(optimizer='adam', loss='mse', metrics=['mae'])

# Train
history_cnn = cnn_model.fit(
    X_train_seq, y_train_seq,
    epochs=50,
    batch_size=32,
    validation_split=0.2,
    callbacks=[early_stopping],
    verbose=0
)

# Evaluate
y_pred_cnn = cnn_model.predict(X_test_seq, verbose=0)
dummy[:, 0] = y_pred_cnn.flatten()
y_pred_cnn_original = scaler_dl.inverse_transform(dummy)[:, 0]

cnn_mae = mean_absolute_error(y_test_lstm_original, y_pred_cnn_original)
cnn_rmse = np.sqrt(mean_squared_error(y_test_lstm_original, y_pred_cnn_original))
cnn_r2 = r2_score(y_test_lstm_original, y_pred_cnn_original)

print("\n1D CNN Model - Test Set:")
print(f"  MAE:  {cnn_mae:.2f}")
print(f"  RMSE: {cnn_rmse:.2f}")
print(f"  R²:   {cnn_r2:.4f}")

---

# 5. Prophet (Facebook's Forecasting Tool)

<div style="
    background-color:#8F0177;
    padding:15px;
    border-radius:8px;
    color:white;
    display:flex;
    align-items:center;
">
    <h3 style="margin:0;">Theory</h3>
</div>

Prophet is designed for business time series with:
- Strong seasonal patterns
- Multiple seasonality (daily, weekly, yearly)
- Holiday effects
- Missing data and outliers

**Model**: y(t) = g(t) + s(t) + h(t) + εₜ
- g(t): Trend
- s(t): Seasonality
- h(t): Holiday effects
- εₜ: Error

In [ ]:
from prophet import Prophet

# Prepare data for Prophet (requires 'ds' and 'y' columns)
prophet_df = pd.DataFrame({
    'ds': df.index,
    'y': df['sales'].values
})

# Split
train_prophet = prophet_df.iloc[:split_idx]
test_prophet = prophet_df.iloc[split_idx:]

print("Prophet data format:")
print(train_prophet.head())

In [ ]:
# Create and fit Prophet model
prophet_model = Prophet(
    yearly_seasonality=True,
    weekly_seasonality=True,
    daily_seasonality=False,
    seasonality_mode='additive'
)

prophet_model.fit(train_prophet)

# Make predictions
future = prophet_model.make_future_dataframe(periods=len(test_prophet), freq='D')
forecast = prophet_model.predict(future)

# Extract test predictions
y_pred_prophet = forecast.iloc[split_idx:]['yhat'].values

# Evaluate
prophet_mae = mean_absolute_error(test_prophet['y'], y_pred_prophet)
prophet_rmse = np.sqrt(mean_squared_error(test_prophet['y'], y_pred_prophet))
prophet_r2 = r2_score(test_prophet['y'], y_pred_prophet)

print("\nProphet Model - Test Set:")
print(f"  MAE:  {prophet_mae:.2f}")
print(f"  RMSE: {prophet_rmse:.2f}")
print(f"  R²:   {prophet_r2:.4f}")

In [ ]:
# Visualize Prophet forecast
fig, ax = plt.subplots(figsize=(14, 6))

# Plot actual data
ax.plot(train_prophet['ds'], train_prophet['y'], label='Train', color='black', linewidth=1.5)
ax.plot(test_prophet['ds'], test_prophet['y'], label='Test (Actual)', color='blue', linewidth=2)

# Plot forecast
forecast_test = forecast.iloc[split_idx:]
ax.plot(forecast_test['ds'], forecast_test['yhat'], label='Prophet Forecast', 
        color='red', linewidth=2, linestyle='--')

# Confidence intervals
ax.fill_between(forecast_test['ds'], 
                forecast_test['yhat_lower'], 
                forecast_test['yhat_upper'],
                alpha=0.2, color='red', label='Confidence Interval')

ax.set_title('Prophet Forecast with Confidence Intervals', fontsize=14, fontweight='bold')
ax.set_xlabel('Date', fontsize=12)
ax.set_ylabel('Sales', fontsize=12)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Visualize components
fig = prophet_model.plot_components(forecast)
plt.tight_layout()
plt.show()

---

# 6. Cross-Validation for Time Series

<div style="
    background-color:#8F0177;
    padding:15px;
    border-radius:8px;
    color:white;
    display:flex;
    align-items:center;
">
    <h3 style="margin:0;">Time Series Cross-Validation</h3>
</div>

Unlike regular CV, we must respect temporal order:

In [ ]:
from sklearn.model_selection import TimeSeriesSplit

# TimeSeriesSplit example
tscv = TimeSeriesSplit(n_splits=5)

print("Time Series Cross-Validation Splits:")
print("="*60)

for i, (train_idx, val_idx) in enumerate(tscv.split(X_train)):
    print(f"\nFold {i+1}:")
    print(f"  Train: {len(train_idx)} samples ({X_train.index[train_idx[0]]} to {X_train.index[train_idx[-1]]})")
    print(f"  Val:   {len(val_idx)} samples ({X_train.index[val_idx[0]]} to {X_train.index[val_idx[-1]]})")

In [ ]:
# Cross-validation with Random Forest
cv_scores = []

for train_idx, val_idx in tscv.split(X_train):
    X_cv_train = X_train.iloc[train_idx]
    X_cv_val = X_train.iloc[val_idx]
    y_cv_train = y_train.iloc[train_idx]
    y_cv_val = y_train.iloc[val_idx]
    
    # Train model
    rf_cv = RandomForestRegressor(n_estimators=100, max_depth=15, random_state=42, n_jobs=-1)
    rf_cv.fit(X_cv_train, y_cv_train)
    
    # Predict and evaluate
    y_pred_cv = rf_cv.predict(X_cv_val)
    rmse = np.sqrt(mean_squared_error(y_cv_val, y_pred_cv))
    cv_scores.append(rmse)

print("\nCross-Validation Results (RMSE):")
print(f"Fold scores: {[f'{score:.2f}' for score in cv_scores]}")
print(f"Mean RMSE: {np.mean(cv_scores):.2f} (+/- {np.std(cv_scores):.2f})")

---

# 7. Final Model Comparison

<div style="
    background-color:#8F0177;
    padding:15px;
    border-radius:8px;
    color:white;
    display:flex;
    align-items:center;
">
    <h3 style="margin:0;">All Models Summary</h3>
</div>


In [ ]:
# Compile all results
final_results = pd.DataFrame([
    {'Model': 'Linear Regression', 'RMSE': test_results[test_results['Model']=='Linear Regression']['RMSE'].values[0]},
    {'Model': 'Ridge', 'RMSE': test_results[test_results['Model']=='Ridge Regression']['RMSE'].values[0]},
    {'Model': 'Lasso', 'RMSE': test_results[test_results['Model']=='Lasso Regression']['RMSE'].values[0]},
    {'Model': 'Decision Tree', 'RMSE': test_results[test_results['Model']=='Decision Tree']['RMSE'].values[0]},
    {'Model': 'Random Forest', 'RMSE': test_results[test_results['Model']=='Random Forest']['RMSE'].values[0]},
    {'Model': 'Gradient Boosting', 'RMSE': test_results[test_results['Model']=='Gradient Boosting']['RMSE'].values[0]},
    {'Model': 'XGBoost', 'RMSE': test_results[test_results['Model']=='XGBoost']['RMSE'].values[0]},
    {'Model': 'LightGBM', 'RMSE': test_results[test_results['Model']=='LightGBM']['RMSE'].values[0]},
    {'Model': 'LSTM', 'RMSE': lstm_rmse},
    {'Model': 'GRU', 'RMSE': gru_rmse},
    {'Model': '1D CNN', 'RMSE': cnn_rmse},
    {'Model': 'Prophet', 'RMSE': prophet_rmse}
]).sort_values('RMSE')

print("\n" + "="*80)
print("FINAL MODEL COMPARISON - ALL APPROACHES")
print("="*80)
print(final_results.to_string(index=False))
print("\n" + "="*80)
print(f"🏆 Best Model: {final_results.iloc[0]['Model']} (RMSE: {final_results.iloc[0]['RMSE']:.2f})")
print("="*80)

In [ ]:
# Visualize final comparison
fig, ax = plt.subplots(figsize=(12, 8))

colors = ['green' if i == 0 else 'steelblue' for i in range(len(final_results))]
final_results.plot(x='Model', y='RMSE', kind='barh', ax=ax, color=colors, legend=False)

ax.set_title('Final Model Comparison - RMSE (Lower is Better)', 
             fontsize=14, fontweight='bold')
ax.set_xlabel('RMSE', fontsize=12)
ax.set_ylabel('Model', fontsize=12)
ax.grid(True, alpha=0.3, axis='x')

# Add value labels
for i, v in enumerate(final_results['RMSE']):
    ax.text(v + 0.5, i, f'{v:.2f}', va='center', fontsize=9)

plt.tight_layout()
plt.show()

---

# Summary and Key Takeaways

<div style="
    background-color:#8F0177;
    padding:15px;
    border-radius:8px;
    color:white;
    display:flex;
    align-items:center;
">
    <h3 style="margin:0;">Feature Engineering Best Practices</h3>
</div>

1. **Lag Features**: Capture autocorrelation (1, 2, 3, 7, 14, 28 days)
2. **Rolling Statistics**: Moving averages, std, min, max (7, 14, 30 day windows)
3. **Date/Time Features**: Day of week, month, quarter, year, is_weekend
4. **Cyclical Encoding**: Use sin/cos for circular features (day of week, month)
5. **Domain Features**: Business-specific indicators (promotions, holidays, events)

<div style="
    background-color:#8F0177;
    padding:15px;
    border-radius:8px;
    color:white;
    display:flex;
    align-items:center;
">
    <h3 style="margin:0;">Model Selection Guide</h3>
</div>

| Model Type | When to Use | Pros | Cons |
|------------|-------------|------|------|
| **Linear Models** | Baseline, linear trends | Fast, interpretable | Limited complexity |
| **Tree-Based** | Non-linear patterns, mixed features | Handle non-linearity, feature importance | Can overfit |
| **Gradient Boosting** | Complex patterns, competitions | High accuracy, handles any data | Slow training, tuning needed |
| **LSTM/GRU** | Long sequences, complex patterns | Learn temporal dependencies | Needs lots of data, slow |
| **Prophet** | Business forecasting, multiple seasonality | Easy to use, handles holidays | Less flexible |

<div style="
    background-color:#8F0177;
    padding:15px;
    border-radius:8px;
    color:white;
    display:flex;
    align-items:center;
">
    <h3 style="margin:0;">Model Training Best Practices</h3>
</div>

1. **Always use chronological splits** - never random for time series
2. **Use TimeSeriesSplit** for cross-validation
3. **Scale features** for neural networks and distance-based models
4. **Handle missing data** before modeling
5. **Monitor for overfitting** - compare train vs validation performance
6. **Create holdout test set** that model never sees during training

<div style="
    background-color:#8F0177;
    padding:15px;
    border-radius:8px;
    color:white;
    display:flex;
    align-items:center;
">
    <h3 style="margin:0;">Evaluation Metrics</h3>
</div>

- **MAE**: Interpretable, less sensitive to outliers
- **RMSE**: Penalizes large errors, same units as target
- **MAPE**: Scale-independent, percentage-based
- **R²**: Proportion of variance explained (0-1)

<div style="
    background-color:#8F0177;
    padding:15px;
    border-radius:8px;
    color:white;
    display:flex;
    align-items:center;
">
    <h3 style="margin:0;">Production Considerations</h3>
</div>

1. **Model Monitoring**: Track performance over time
2. **Retraining**: Update models regularly with new data
3. **Feature Storage**: Save feature engineering pipeline
4. **A/B Testing**: Compare models in production
5. **Fallback Strategy**: Have simple baseline ready
6. **Explainability**: Use SHAP/LIME for model interpretation

<div style="
    background-color:#8F0177;
    padding:15px;
    border-radius:8px;
    color:white;
    display:flex;
    align-items:center;
">
    <h3 style="margin:0;">Recommended Workflow</h3>
</div>

```python
# 1. Start with simple baselines
naive, moving_average, exponential_smoothing

# 2. Try classical statistical models
ARIMA, SARIMA

# 3. Feature engineering + ML
Random Forest, XGBoost, LightGBM

# 4. Try specialized tools
Prophet (if multiple seasonality)

# 5. Deep learning (if needed)
LSTM, GRU (if large dataset and complex patterns)

# 6. Ensemble best models
Combine predictions from multiple models
```

<div style="
    background-color:#8F0177;
    padding:15px;
    border-radius:8px;
    color:white;
    display:flex;
    align-items:center;
">
    <h3 style="margin:0;">Common Pitfalls to Avoid</h3>
</div>

❌ Using random train-test split
❌ Data leakage (using future information)
❌ Ignoring seasonality
❌ Not handling missing values properly
❌ Overfitting on small datasets
❌ Not validating on multiple time periods

<div style="
    background-color:#8F0177;
    padding:15px;
    border-radius:8px;
    color:white;
    display:flex;
    align-items:center;
">
    <h3 style="margin:0;">Further Learning</h3>
</div>

- **Ensemble Methods**: Combine multiple models
- **Attention Mechanisms**: Transformer-based models
- **Transfer Learning**: Use pre-trained models
- **Probabilistic Forecasting**: Prediction intervals
- **Multivariate Methods**: VAR, VECM
- **Online Learning**: Update models in real-time

---

# Conclusion

You now have a complete toolkit for time series analysis:

**Part 1**: Statistical foundations (ARIMA, SARIMA)
**Part 2**: Pandas operations and data manipulation
**Part 3**: Machine learning and deep learning methods

The best approach depends on:
- Dataset size
- Pattern complexity
- Computational resources
- Interpretability requirements
- Production constraints

Start simple, iterate, and always validate carefully!

---

*End of Part 3 - Complete Time Series Analysis Course*